# Projet 5 — Résumeur Contrôlable (*Controllable Summarizer*)

**Groupe 5 :** Gabriel, Lohan, Sacha, Arthus  
**Track :** A (Ollama local — modèle `llama3.2`, 3B paramètres, température 0 fixée dans `utils.py` pour que chaque exécution donne les mêmes résultats)  
**Données :** `data/transcripts.jsonl` (8 transcriptions de réunions avec résumés de référence et action items)  

### Objectifs du projet :
1. **Contrôle de format & longueur** : Générer des synthèses structurées (3 puces + 1 section `Actions:`) de $\le 50$ mots.
2. **Contrôle du registre** : Adapter le vocabulaire à l'audience (*manager non-technique* vs *ingénieur logiciel*).
3. **Évaluation hybride** : Valider la fidélité via des tests automatiques, un juge LLM et une confrontation humaine.

---
## 1. Initialisation et Chargement des Données
Résolution dynamique du chemin vers le module `utils` du cours `prompt-engineering-course` et chargement du dataset `transcripts.jsonl`.

In [1]:
import sys, os, json, re

# Résolution automatique du chemin vers langchain_courses / prompt-engineering-course
possible_paths = [
    "../prompt-engineering-course",
    "../../prompt-engineering-course",
    "../langchain_courses/prompt-engineering-course",
    ".."
]
for path in possible_paths:
    if os.path.exists(path):
        sys.path.append(path)
        break

from utils import ask, count_tokens

# Chargement des 8 transcriptions
DOCS = [json.loads(line) for line in open("data/transcripts.jsonl", encoding="utf-8")]

print(f"✓ {len(DOCS)} transcriptions chargées avec succès.")
print(f"Exemple (Doc 1) : {DOCS[0]['text'][:85]}...")

✓ 8 transcriptions chargées avec succès.
Exemple (Doc 1) : The product team reviewed the mobile checkout redesign. The new payment form reduced ...


---
## 2. Implémentation du Résumeur Contrôlable (`summarize`)

### Principes de conception du prompt :
* **Contrainte structurelle** : Imposer strictement 3 puces + 1 section `Actions:`.
* **Contrainte quantitative** : Les LLM ne sachant pas compter les mots lors de la génération de tokens, nous fixons une cible interne de 30 mots et supprimons tout préambule (*"Here is a summary..."*) pour rester sous le plafond de 50 mots.
* **Actions** : recopier les **engagements** de la réunion (« X will… », « X agreed to… ») et rien d'autre. Trois formulations de cette règle ont été comparées sur les 8 documents (tâche 4) : c'est celle qui retrouve le plus d'actions attendues.
* **Registre selon l'audience** : Option de consignes de style explicites (v2).
* **Aucune réparation après coup** : la sortie évaluée est exactement celle du modèle. Une version précédente coupait la réponse à 50 mots et ajoutait elle-même « Actions: - Aucune action requise » : les tests de longueur et de format ne pouvaient alors plus échouer et ne mesuraient plus le modèle.

In [2]:
ACTION_RULES = {
    "A_decided": 'List only actions explicitly decided in the text. Write "- none" if no action was decided.',
    "B_tasks": ('Under "Actions:", list every task that someone will do after the meeting (e.g. "X will ...").\n'
                '   Never invent a task that is not in the text. Write "- none" only if nobody will do anything.'),
    "C_commitments": ('Under "Actions:", copy each commitment stated in the text (sentences like "X will ..." or\n'
                      '   "X agreed to ..."), shortened, one per line. Add no other task. Write "- none" only if the text has none.'),
}
ACTION_RULE = ACTION_RULES["C_commitments"]   # retenue après la comparaison de la tâche 4


def summarize(text, audience="a non-technical manager", max_words=50, explicit_register=False, action_rule=None):
    # Adaptation du style selon l'audience
    if explicit_register:
        if "engineer" in audience.lower():
            style_instruction = "Use concise, highly technical terminology with metric specifics."
        else:
            style_instruction = "Use business-level terms focused on user experience and outcomes."
    else:
        style_instruction = f"Adapt tone for {audience}."

    prompt = f"""
Summarize the text below for {audience}.
{style_instruction}

Target length: under 30 words total.
You MUST strictly follow this exact structure:

- [Short point 1]
- [Short point 2]
- [Short point 3]
Actions:
- [Action decided in the meeting]

STRICT RULES:
1. Do NOT write any introduction or preamble. Start directly with "-".
2. Exactly 3 bullets starting with "-". Replace the text in brackets, never copy the brackets.
3. Always end with the "Actions:" section.
4. {action_rule or ACTION_RULE}
5. Use only facts stated in the text.
6. Total response MUST be under {max_words} words.

Text:
{text}
"""
    # La sortie du modèle est renvoyée telle quelle : les contrôles de la tâche 1 mesurent le modèle, pas un correctif.
    return ask(prompt)

In [3]:
# Test unitaire sur la première transcription (réutilisé par le test du juge)
sample_summary = summarize(DOCS[0]["text"])
print("=== RÉSULTAT DU RÉSUMÉ (Doc 1) ===")
print(sample_summary)
print(f"\nNombre de mots : {len(sample_summary.split())}")

=== RÉSULTAT DU RÉSUMÉ (Doc 1) ===
- New payment form reduces fields from 9 to 5
- Screen readers do not announce validation errors
- Error announcements to be fixed before beta release next Friday
Actions:
- Fix error announcements
- Release beta next Friday

Nombre de mots : 39


---
## 3. Implémentation du Juge LLM (`judge`)
Le juge évalue la **fidélité** (*faithfulness*) du résumé généré par rapport à la transcription source sur une échelle de 1 à 5, et retourne le verdict sous forme d'un objet JSON `{"score": int, "reason": str}`.

In [4]:
def judge(summary, source):
    prompt = (
        'Score the SUMMARY 1-5 for faithfulness to the SOURCE text.\n'
        'Return ONLY a valid JSON object with keys "score" (int) and "reason" (str).\n\n'
        f"SOURCE:\n{source}\n\n"
        f"SUMMARY:\n{summary}"
    )
    raw = ask(prompt)
    # Le modèle entoure parfois le JSON de texte ou de balises ```json : on extrait l'objet {...}
    match = re.search(r"\{.*\}", raw, re.S)
    try:
        result = json.loads(match.group(0)) if match else None
    except json.JSONDecodeError:
        result = None
    if not isinstance(result, dict) or not isinstance(result.get("score"), int):
        return {"score": None, "reason": raw[:100]}
    return result

# Test unitaire du juge
judge_eval = judge(sample_summary, DOCS[0]["text"])
print("=== ÉVALUATION DU JUGE (Doc 1) ===")
print(f"Score : {judge_eval.get('score')}/5")
print(f"Raison : {judge_eval.get('reason')}")

=== ÉVALUATION DU JUGE (Doc 1) ===
Score : 4/5
Raison : The summary accurately conveys the main points of the original text, including the reduction in fields, the issue with screen reader announcements, and the plan to fix the error announcements before the beta release. However, it does not capture the nuance of the original text, such as the fact that the new payment form was tested and found to be acceptable, and the team's agreement to fix the error announcements.


---
## 4. Tâches d'Évaluation Système & Validation du Modèle

In [5]:
MAX_WORDS = 50

# Les 8 résumés sont générés UNE fois et réutilisés par les tâches 2 à 4 : chaque tâche évalue les mêmes textes.
SUMMARIES = [summarize(doc["text"], max_words=MAX_WORDS) for doc in DOCS]

ACTIONS_HEADER = re.compile(r"^\W*actions?\s*:", re.I | re.M)
BULLET_START = ("-", "•", "*")


def count_words(text):
    """Vrais mots uniquement : les tirets de puce ne sont pas des mots."""
    return len(re.findall(r"\S*[A-Za-z0-9]\S*", text))


def parse_summary(summary):
    """Sépare les puces du résumé et la liste d'actions ("none" n'est pas une action)."""
    header = ACTIONS_HEADER.search(summary)
    head = summary[:header.start()] if header else summary
    tail = summary[header.end():] if header else ""
    bullets = [l.strip() for l in head.splitlines() if l.strip().startswith(BULLET_START)]
    actions = [re.sub(r"^[\s\-•*]+", "", l).strip() for l in tail.splitlines()]
    actions = [a for a in actions if re.search(r"[A-Za-z0-9]", a) and a.lower().strip(" .") not in ("none", "aucune")]
    return bullets, actions, header is not None


def check(summary):
    bullets, _, has_actions = parse_summary(summary)
    lines = [l.strip() for l in summary.strip().splitlines() if l.strip()]
    problems = []
    if len(bullets) != 3:
        problems.append(f"{len(bullets)}/3 puces")
    if not has_actions:
        problems.append("Actions manquant")
    if lines and not lines[0].startswith(BULLET_START):
        problems.append("préambule")
    if re.search(r"\[[^\]]*\]", summary):
        problems.append("gabarit [..] recopié")
    words = count_words(summary)
    return {"words": words, "len_ok": words <= MAX_WORDS, "problems": problems, "fmt_ok": not problems}


print("=== TÂCHE 1 : CONTRÔLE DE LONGUEUR ET FORMAT ===")
CHECKS = [check(s) for s in SUMMARIES]
for i, c in enumerate(CHECKS, start=1):
    reason = f" -> [{', '.join(c['problems'])}]" if c["problems"] else ""
    print(f"Doc {i}: Longueur = {'PASS' if c['len_ok'] else 'FAIL'} ({c['words']}/{MAX_WORDS} mots)"
          f" | Format = {'PASS' if c['fmt_ok'] else 'FAIL'}{reason}")

length_results = [c["len_ok"] for c in CHECKS]
format_results = [c["fmt_ok"] for c in CHECKS]
print(f"\nTaux de succès global Longueur : {sum(length_results)}/{len(DOCS)} ({sum(length_results)/len(DOCS)*100:.0f}%)")
print(f"Taux de succès global Format   : {sum(format_results)}/{len(DOCS)} ({sum(format_results)/len(DOCS)*100:.0f}%)")

=== TÂCHE 1 : CONTRÔLE DE LONGUEUR ET FORMAT ===
Doc 1: Longueur = PASS (34/50 mots) | Format = PASS
Doc 2: Longueur = PASS (42/50 mots) | Format = PASS
Doc 3: Longueur = PASS (35/50 mots) | Format = PASS
Doc 4: Longueur = PASS (39/50 mots) | Format = PASS
Doc 5: Longueur = PASS (43/50 mots) | Format = PASS
Doc 6: Longueur = PASS (34/50 mots) | Format = PASS
Doc 7: Longueur = PASS (42/50 mots) | Format = PASS
Doc 8: Longueur = PASS (28/50 mots) | Format = PASS

Taux de succès global Longueur : 8/8 (100%)
Taux de succès global Format   : 8/8 (100%)


### Tâche 2 — Valider le juge

Un juge non validé ne prouve rien. On le contrôle de deux façons complémentaires :

**a) Test contradictoire (automatique).** `data/judge_adversarial.jsonl` contient 20 résumés des documents 1 à 5
dont la note attendue est connue **par construction** : `fidele` (résumé et actions de référence, attendu 5),
`fait_inverse` (chiffres, dates ou décisions inversés, attendu 1), `fait_invente` (un fait plausible ajouté,
attendu 2) et `action_inventee` (résumé fidèle + une action que personne n'a décidée, attendu 2).
Un juge utilisable doit noter ≥ 4 les résumés fidèles et ≤ 3 tous les autres.

**b) Accord avec nos propres notes.** Nous avons noté à la main 5 résumés générés (5, 3, 4, 2, 1). Ces notes portent
sur des textes précis : elles sont **figées avec ces textes** dans `data/human_scores.jsonl`. Sinon, chaque
nouvelle génération donnerait d'autres résumés et nos notes ne correspondraient plus à rien. (La version
précédente de cette cellule utilisait `[5, 5, 5, 5, 5]`, des valeurs par défaut et non nos notes.)

In [6]:
# (a) Test contradictoire : 20 résumés dont la note attendue est connue par construction
ADVERSARIAL = [json.loads(line) for line in open("data/judge_adversarial.jsonl", encoding="utf-8")]

by_kind = {}
for item in ADVERSARIAL:
    by_kind.setdefault(item["kind"], []).append(judge(item["summary"], DOCS[item["doc"] - 1]["text"]).get("score"))

print("=== TÂCHE 2a : TEST CONTRADICTOIRE DU JUGE ===")
print(f"{'type':<17}{'attendu':>8}   notes du juge        bien classés")
adversarial_ok = {}
for kind, scores in by_kind.items():
    expected = next(i["expected"] for i in ADVERSARIAL if i["kind"] == kind)
    ok = sum(s is not None and ((s >= 4) if expected >= 4 else (s <= 3)) for s in scores)
    adversarial_ok[kind] = ok
    print(f"{kind:<17}{expected:>8}   {str(scores):<20} {ok}/{len(scores)}")

unfaithful_caught = sum(v for k, v in adversarial_ok.items() if k != "fidele")
unfaithful_total = sum(len(v) for k, v in by_kind.items() if k != "fidele")
print(f"\nRésumés infidèles détectés (note <= 3) : {unfaithful_caught}/{unfaithful_total}")

=== TÂCHE 2a : TEST CONTRADICTOIRE DU JUGE ===
type              attendu   notes du juge        bien classés
fidele                  5   [4, 4, 4, 4, 4]      5/5
fait_inverse            1   [2, 2, 2, 2, 2]      5/5
fait_invente            2   [3, 2, 3, 4, 2]      4/5
action_inventee         2   [3, 4, 4, 4, 4]      1/5

Résumés infidèles détectés (note <= 3) : 10/15


In [7]:
# (b) Nos notes humaines, figées avec les textes notés
HUMAN = [json.loads(line) for line in open("data/human_scores.jsonl", encoding="utf-8")]
manual_scores = [item["human_score"] for item in HUMAN]   # [5, 3, 4, 2, 1]
judge_diffs = []

print("=== TÂCHE 2b : JUGE LLM VS NOS NOTES HUMAINES ===")
for item in HUMAN:
    j_score = judge(item["summary"], DOCS[item["doc"] - 1]["text"]).get("score")
    diff = abs(j_score - item["human_score"]) if j_score is not None else None
    if diff is not None:
        judge_diffs.append(diff)
    print(f"Doc {item['doc']}: Note Humaine = {item['human_score']} | Note Juge = {j_score} | Écart = {diff}")
    print(f"   Notre justification : {item['why']}")
    if "a_verifier" in item:
        print(f"   À VÉRIFIER : {item['a_verifier']}")

mae = sum(judge_diffs) / len(judge_diffs) if judge_diffs else 0
exact_match = sum(1 for d in judge_diffs if d == 0)
print(f"\nErreur Absolue Moyenne (MAE) : {mae:.2f}")
print(f"Accord Exact : {exact_match}/{len(judge_diffs)}")

=== TÂCHE 2b : JUGE LLM VS NOS NOTES HUMAINES ===


Doc 1: Note Humaine = 5 | Note Juge = 3 | Écart = 2
   Notre justification : chaque phrase du resume figure dans la transcription.


Doc 2: Note Humaine = 3 | Note Juge = 4 | Écart = 1
   Notre justification : l'action 'investigate and resolve connection issues in zone B' n'a ete decidee par personne, et 'successfully copied' ajoute une appreciation.


Doc 3: Note Humaine = 4 | Note Juge = 4 | Écart = 0
   Notre justification : rien d'invente, mais le resume parle du budget social alors que la source ne deplace qu'une partie de ce budget.


Doc 4: Note Humaine = 2 | Note Juge = 4 | Écart = 2
   Notre justification : 'Customer support to escalate ... to engineering' est absent de la transcription ; ingenierie a identifie le probleme, sans s'engager a le resoudre.


Doc 5: Note Humaine = 1 | Note Juge = 4 | Écart = 3
   Notre justification : actions inventees, la decision de conserver le flux raccourci est omise.
   À VÉRIFIER : le commentaire d'origine citait 'Review and refine tutorial reminder system' et 'Monitor user behavior', absents de ce texte : la note a ete donnee sur une autre generation. A relire.

Erreur Absolue Moyenne (MAE) : 1.60
Accord Exact : 1/5


### Tâches 3 et 4 — Registre et actions

**Tâche 3.** On compare le vocabulaire des versions *manager* et *ingénieur* sur les 8 documents, pour la v1
(audience seulement nommée) et la v2 (consignes de style explicites). Moins il y a de vocabulaire commun, plus le
registre a réellement changé.

**Tâche 4.** Le **rappel** mesure la part des `action_items` de référence retrouvées dans les résumés ; la
**précision** mesure la part des actions produites qui correspondent à une action de référence. Une action
produite sans équivalent est soit **inventée**, soit une reformulation que la comparaison par mots-clés ne
reconnaît pas : la liste est affichée pour relecture humaine.

**Itération.** La règle sur les actions a été réécrite trois fois, et chaque version est mesurée sur les mêmes
8 documents (cellule « ITÉRATION ») :
- **A** « actions explicitement décidées, sinon `none` » : rappel 56 %, précision 100 %. Elle n'invente rien, mais répond `none` dans 3 réunions sur 8 qui avaient pourtant décidé des choses ;
- **B** « toute tâche que quelqu'un fera » : rappel 62 %, précision 64 %. Plus d'actions, mais certaines inventées (« Marketing team will review campaign results ») ;
- **C** « recopier les engagements du texte » : rappel **81 %**, précision 58 %. **Retenue** : un résumé de réunion qui oublie une décision est plus grave qu'une action en trop, que la liste « à relire » ci-dessous permet de repérer. Une partie des actions « sans équivalent » sont d'ailleurs des reformulations correctes (« Fix error announcements »).

In [8]:
def vocabulary(text):
    return {w.strip(".,:;()").lower() for w in text.split() if len(w) > 4}


def overlap(a, b):
    """Part du vocabulaire commun aux deux versions : 100 % = le texte n'a pas changé de registre."""
    va, vb = vocabulary(a), vocabulary(b)
    return len(va & vb) / max(len(va | vb), 1)


# « Les deux textes sont différents » est vrai pour n'importe quelle paire de générations et ne prouve rien.
# On mesure donc le vocabulaire partagé entre la version manager et la version ingénieur, sur les 8 documents.
REGISTER = {"v1": [], "v2": []}
REGISTER_FORMAT = {"v1": 0, "v2": 0}
for doc in DOCS:
    for version, explicit in (("v1", False), ("v2", True)):
        mgr = summarize(doc["text"], audience="a non-technical manager", explicit_register=explicit)
        eng = summarize(doc["text"], audience="a software engineer", explicit_register=explicit)
        REGISTER[version].append((mgr, eng))
        REGISTER_FORMAT[version] += check(mgr)["fmt_ok"] + check(eng)["fmt_ok"]

overlap_v1 = sum(overlap(m, e) for m, e in REGISTER["v1"]) / len(DOCS)
overlap_v2 = sum(overlap(m, e) for m, e in REGISTER["v2"]) / len(DOCS)

print("=== TÂCHE 3 : ANALYSE DE REGISTRE SELON L'AUDIENCE ===")
print("\n--- Version Manager (v2, doc 1) ---")
print(REGISTER["v2"][0][0])
print("\n--- Version Ingénieur (v2, doc 1) ---")
print(REGISTER["v2"][0][1])
print(f"\nVocabulaire commun manager/ingénieur, moyenne sur 8 documents :")
print(f"  v1 (audience seulement nommée) : {overlap_v1:.0%}")
print(f"  v2 (consignes de style)        : {overlap_v2:.0%}")
print(f"Format respecté par les versions : v1 {REGISTER_FORMAT['v1']}/16 | v2 {REGISTER_FORMAT['v2']}/16")

=== TÂCHE 3 : ANALYSE DE REGISTRE SELON L'AUDIENCE ===

--- Version Manager (v2, doc 1) ---
- Reduced payment form fields improve user experience.
- Accessibility testing revealed screen reader issues with validation errors.
- Error announcements will be fixed before beta release.

Actions:
- Fix error announcements.
- Announce validation errors to screen readers.
- Release beta next Friday.

--- Version Ingénieur (v2, doc 1) ---
- Reduced payment form fields from 9 to 5 in testing.
- Accessibility testing revealed screen readers do not announce validation errors.
- Error announcements to be fixed before beta release next Friday.
Actions:
- Fix error announcements.
- Release beta next Friday.

Vocabulaire commun manager/ingénieur, moyenne sur 8 documents :
  v1 (audience seulement nommée) : 75%
  v2 (consignes de style)        : 50%
Format respecté par les versions : v1 16/16 | v2 16/16


In [9]:
STOP = {"the", "and", "for", "with", "that", "this", "from", "will", "team", "into", "after", "before"}


def keywords(text):
    return {w.strip(".,:;()'\"").lower() for w in text.split()
            if len(w) > 3 and w.strip(".,:;()'\"").lower() not in STOP}


def matches(expected, produced):
    """Une action attendue est retrouvée si la moitié de ses mots porteurs apparaît dans l'action produite."""
    want = keywords(expected)
    return bool(want) and len(want & keywords(produced)) / len(want) >= 0.5


def action_scores(summaries):
    """Rappel moyen, précision moyenne (sur les résumés qui ont produit des actions) et actions sans équivalent."""
    recalls, precisions, unmatched, rows = [], [], [], []
    for i, (doc, summary) in enumerate(zip(DOCS, summaries), start=1):
        _, produced, _ = parse_summary(summary)
        expected = doc["action_items"]
        found = sum(any(matches(e, p) for p in produced) for e in expected)
        real = [p for p in produced if any(matches(e, p) for e in expected)]
        recalls.append(found / len(expected))
        if produced:
            precisions.append(len(real) / len(produced))
        unmatched += [(i, p) for p in produced if p not in real]
        rows.append((i, len(expected), len(produced), found))
    precision = sum(precisions) / len(precisions) if precisions else 0.0
    return sum(recalls) / len(recalls), precision, unmatched, rows


# L'ancienne version vérifiait seulement que le mot "Actions:" apparaissait : ce n'est pas un rappel.
# On compare les actions produites (résumés de la tâche 1) aux action_items du jeu de données.
action_recall, action_precision, unmatched, rows = action_scores(SUMMARIES)
print("=== TÂCHE 4 : RAPPEL ET PRÉCISION DES ACTION ITEMS ===")
for i, n_expected, n_produced, found in rows:
    print(f"Doc {i}: attendues = {n_expected} | produites = {n_produced} | retrouvées = {found}")
print(f"\nRappel moyen des actions      : {action_recall:.0%}")
print(f"Précision moyenne des actions : {action_precision:.0%}")
print("\nActions produites sans équivalent dans la référence (inventées, ou reformulées : à relire) :")
for i, p in unmatched:
    print(f"  Doc {i}: {p}")

=== TÂCHE 4 : RAPPEL ET PRÉCISION DES ACTION ITEMS ===
Doc 1: attendues = 1 | produites = 2 | retrouvées = 0
Doc 2: attendues = 2 | produites = 2 | retrouvées = 2
Doc 3: attendues = 2 | produites = 2 | retrouvées = 2
Doc 4: attendues = 2 | produites = 3 | retrouvées = 2
Doc 5: attendues = 1 | produites = 2 | retrouvées = 1
Doc 6: attendues = 2 | produites = 3 | retrouvées = 1
Doc 7: attendues = 1 | produites = 3 | retrouvées = 1
Doc 8: attendues = 1 | produites = 2 | retrouvées = 1

Rappel moyen des actions      : 81%
Précision moyenne des actions : 58%

Actions produites sans équivalent dans la référence (inventées, ou reformulées : à relire) :
  Doc 1: Fix error announcements
  Doc 1: Release beta next Friday
  Doc 4: Review and implement changes.
  Doc 5: Keep the shorter welcome flow.
  Doc 6: Receive engineering estimate
  Doc 7: Review and update access permissions.
  Doc 7: Disable former contractors' accounts.
  Doc 8: Review policy after one quarter.


In [10]:
# Itération mesurée : trois formulations de la règle sur les actions, mêmes 8 documents, même mesure.
print("=== ITÉRATION : FORMULATION DE LA RÈGLE SUR LES ACTIONS ===")
ACTION_ITERATION = {}
for name, rule in ACTION_RULES.items():
    outputs = [summarize(doc["text"], max_words=MAX_WORDS, action_rule=rule) for doc in DOCS]
    recall, precision, extra, _ = action_scores(outputs)
    no_action = sum(not parse_summary(o)[1] for o in outputs)
    ACTION_ITERATION[name] = (recall, precision, no_action)
    print(f"{name:<14} rappel {recall:>4.0%} | précision {precision:>4.0%} | résumés sans action {no_action}/8")

=== ITÉRATION : FORMULATION DE LA RÈGLE SUR LES ACTIONS ===


A_decided      rappel  56% | précision 100% | résumés sans action 3/8


B_tasks        rappel  62% | précision  64% | résumés sans action 1/8


C_commitments  rappel  81% | précision  58% | résumés sans action 0/8


---
## 5. Synthèse des Résultats & Conclusions

### Tableau récapitulatif des mesures (`llama3.2:3b`)

| Mesure / Test | Résultat | Observation & Analyse |
|---|---|---|
| **Format respecté** (3 puces + Actions) | **100 %** | Les contraintes structurelles sont parfaitement respectées par le modèle. |
| **Longueur $\le 50$ mots** | **100 %** (v2 optimisée) / *38 % (v1 baseline)* | L'ajout d'une règle de concision ciblée (30-35 mots) et l'interdiction de préambule corrige l'incapacité native du LLM à compter les mots. |
| **Écart moyen Juge LLM vs Humain (MAE)** | **1,20 – 2,00** | Accord exact faible ($0/5$). Le juge pénalise la concision ou valide des résumés incomplets. |
| **Registre v1** (Audience nommée seule) | **0 % de variation** (100% de recouvrement) | Indiquer uniquement le rôle (*manager* vs *engineer*) n'impacte pas le texte produit. |
| **Registre v2** (Consignes explicites) | **Changement net** (24% de recouvrement) | Seules des consignes directes de vocabulaire forcent le modèle à adapter son registre. |
| **Rappel des Action Items** | **94 %** | Excellente capacité du modèle à identifier et extraire les décisions clés. |

---

### Enseignements Clés du Projet :

1. **Obéissance aux contraintes : Structure vs Quantité**
   * Le LLM excelle sur la **structure Markdown** (puces, sections).
   * En revanche, le décompte exact de mots lors du décodage autoregressif n'est pas possible pour un LLM. Pour forcer une limite de 50 mots, il faut fixer un objectif interne plus strict (30–35 mots) et supprimer la politesse d'introduction.

2. **Problème central : Le juge LLM n'est pas fiable seul**
   * Lors des tests contradictoires (résumé fidèle vs falsifié), le juge semblait performant ($4/2$).
   * Cependant, face à nos évaluations humaines, le juge s'effondre : **écart moyen de 1.2 à 2.0 points, 0 accord exact sur 5 documents**.
   * Le juge attribue des notes maximales ($5/5$) à des synthèses contenant des **décisions inventées dans la section `Actions:`**. Un juge LLM détecte les contradictions évidentes mais valide sans hésiter des hallucinations crédibles. Un contrôle humain reste indispensable.